# Chapter 3: Multiagent Search

```{admonition} Learning Objectives
:class: tip
- Understand game theory and adversarial search
- Master AND-OR search trees
- Implement Minimax algorithm with perfect play analysis
- Optimize search with Alpha-Beta pruning
- Apply Monte Carlo Tree Search (MCTS)
- Design evaluation functions for complex games
- Build complete game-playing agents
- Compare deductive vs inductive game-playing approaches
```

```{epigraph}
In games, the opponent is part of the environment but with their own intelligence.

-- Stuart Russell & Peter Norvig
```

## 3.1 Introduction

Multiagent environments involve multiple entities making decisions that affect each other. This chapter focuses on **adversarial search** where agents have conflicting goals.

### Single-Agent vs. Multi-Agent Search

**Single-Agent (Previous Chapter):**
- Agent vs. nature
- Complete control over outcomes
- Solution is a sequence of actions
- OR nodes only (agent chooses)

**Multi-Agent (This Chapter):**
- Agent vs. intelligent opponent(s)
- Cannot control opponent's actions
- Solution is a **strategy** (contingent plan)
- AND-OR nodes (agent chooses, opponent responds)

### Game Theory Basics

**Key Concepts:**

1. **Players**: Decision-makers (usually 2 in this chapter)
2. **States**: Configurations of the game
3. **Actions**: Legal moves for each player
4. **Terminal Test**: Is the game over?
5. **Utility Function**: Payoff at terminal states

**Game Types:**

- **Zero-Sum**: One player's gain is another's loss (chess, checkers)
- **Perfect Information**: All state info visible (chess) vs. imperfect (poker)
- **Deterministic**: No chance involved vs. stochastic (backgammon)
- **Turn-Taking**: Players alternate moves

This chapter focuses on **zero-sum, perfect information, deterministic, turn-taking games**.

### Why Games Matter in AI

Games are important for AI research because:

1. **Well-Defined**: Clear rules, objectives, states
2. **Benchmarks**: Easy to measure performance
3. **Challenging**: Require strategic thinking, planning ahead
4. **Generalizable**: Techniques apply to real-world adversarial scenarios
5. **Historical Milestones**: Deep Blue (chess), AlphaGo (Go), AlphaZero

**Real-World Applications:**
- Military strategy and planning
- Business competition and pricing
- Security and defense (attacker vs. defender)
- Negotiation and auctions
- Autonomous systems in competitive environments

In [ ]:
# Essential imports
import numpy as np
import math
import random
import time
from typing import List, Optional, Tuple, Any
from abc import ABC, abstractmethod
from collections import defaultdict
from copy import deepcopy

# For visualization (optional)
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("Matplotlib not available - visualizations disabled")

print('✓ Libraries imported successfully')

## 3.2 Game State Representation

We need a framework to represent games uniformly. This allows us to write game-playing algorithms once and apply them to any game.

### Game State Interface

Every game must implement:

1. **get_legal_actions(player)**: Return valid moves
2. **get_successor(action)**: Return resulting state after action
3. **is_terminal()**: Check if game is over
4. **utility(player)**: Return payoff for terminal state
5. **current_player()**: Whose turn is it?

Optional but useful:
- **display()**: Visualize the game state
- **copy()**: Create independent copy
- **hash()**: For transposition tables

In [ ]:
class GameState(ABC):
    """
    Abstract base class for two-player game states.
    Player 1 (MAX) tries to maximize utility.
    Player -1 (MIN) tries to minimize utility.
    """
    
    @abstractmethod
    def get_legal_actions(self, player: int) -> List[Any]:
        """Return list of legal actions for the player."""
        pass
    
    @abstractmethod
    def get_successor(self, action: Any):
        """Return new state after applying action."""
        pass
    
    @abstractmethod
    def is_terminal(self) -> bool:
        """Check if game is over."""
        pass
    
    @abstractmethod
    def utility(self, player: int) -> float:
        """Return utility value for terminal state from player's perspective."""
        pass
    
    @abstractmethod
    def current_player(self) -> int:
        """Return current player (1 or -1)."""
        pass
    
    def display(self):
        """Display the game state (optional)."""
        print(self)


print('✓ GameState abstract class defined')

## 3.3 Minimax Algorithm

Minimax is the fundamental algorithm for two-player zero-sum games with perfect information.

### Core Idea

**Assumptions:**
1. Both players play optimally
2. MAX tries to maximize utility
3. MIN tries to minimize utility
4. Perfect information (both see full state)

**Minimax Value**:

$$\text{MINIMAX}(s) = \begin{cases}
\text{UTILITY}(s) & \text{if } s \text{ is terminal} \\
\max_{a} \text{MINIMAX}(\text{RESULT}(s, a)) & \text{if } s \text{ is MAX's turn} \\
\min_{a} \text{MINIMAX}(\text{RESULT}(s, a)) & \text{if } s \text{ is MIN's turn}
\end{cases}$$

### Algorithm Explanation

1. **Terminal State**: Return utility directly
2. **MAX's Turn**: Choose action that leads to state with highest minimax value
3. **MIN's Turn**: Choose action that leads to state with lowest minimax value
4. **Recursion**: Compute minimax value of successor states

### Properties

- **Complete**: Yes (for finite games)
- **Optimal**: Yes (against optimal opponent)
- **Time Complexity**: O(b^m) where b = branching factor, m = max depth
- **Space Complexity**: O(bm) with DFS

### Game Tree Example

```
        MAX
       /   \
      3     12
     / \   / \
    3   12 8  14
```

- MIN chooses minimum of children
- MAX chooses maximum of children
- MAX should choose right branch (value 12)

In [ ]:
def minimax(state: GameState, 
            depth: int = 0,
            maximizing: bool = True,
            eval_fn: Optional[callable] = None,
            max_depth: float = float('inf')) -> Tuple[float, Optional[Any]]:
    """
    Minimax algorithm implementation.
    
    Args:
        state: Current game state
        depth: Current depth in tree
        maximizing: True if MAX's turn, False if MIN's
        eval_fn: Evaluation function for non-terminal states at depth limit
        max_depth: Maximum search depth
    
    Returns:
        (value, best_action) tuple
    """
    # Terminal state or depth limit
    if state.is_terminal():
        player = 1 if maximizing else -1
        return state.utility(player), None
    
    if depth >= max_depth:
        if eval_fn:
            return eval_fn(state), None
        return 0, None
    
    player = state.current_player()
    legal_actions = state.get_legal_actions(player)
    
    if not legal_actions:
        # No legal moves - treat as terminal
        return state.utility(player), None
    
    if maximizing:
        # MAX player - maximize value
        max_value = float('-inf')
        best_action = None
        
        for action in legal_actions:
            successor = state.get_successor(action)
            value, _ = minimax(successor, depth + 1, False, eval_fn, max_depth)
            
            if value > max_value:
                max_value = value
                best_action = action
        
        return max_value, best_action
    else:
        # MIN player - minimize value
        min_value = float('inf')
        best_action = None
        
        for action in legal_actions:
            successor = state.get_successor(action)
            value, _ = minimax(successor, depth + 1, True, eval_fn, max_depth)
            
            if value < min_value:
                min_value = value
                best_action = action
        
        return min_value, best_action


print('✓ Minimax algorithm implemented')

## 3.4 Alpha-Beta Pruning

Alpha-Beta pruning is an optimization of Minimax that eliminates branches that cannot influence the final decision.

### Key Idea

**Observation**: We don't need to evaluate every node to find the minimax value.

**Parameters:**
- **α (alpha)**: Best value MAX can guarantee so far (lower bound)
- **β (beta)**: Best value MIN can guarantee so far (upper bound)

**Pruning Condition**: When α ≥ β, we can prune (stop searching that branch).

### How It Works

1. **MAX Node**: Updates α with maximum value found
   - If α ≥ β, MIN won't choose this branch (prune)

2. **MIN Node**: Updates β with minimum value found
   - If β ≤ α, MAX won't choose this branch (prune)

### Example

```
        MAX (α=-∞, β=+∞)
       /   \
      3     ?
     / \   / \
    3  12  2  X  <- Can prune X! (MIN already found 2 < 3)
```

### Effectiveness

**Best Case**: O(b^(d/2)) - doubles effective search depth!
- Occurs with perfect move ordering
- Allows searching to depth 2d instead of d

**Worst Case**: O(b^d) - same as Minimax
- Occurs with worst move ordering

**Average Case**: O(b^(3d/4))

**Move Ordering Importance:**
Good move ordering is crucial:
- Try promising moves first
- Use heuristics (e.g., captures before quiet moves in chess)
- Use transposition tables
- Try moves from previous iteration (iterative deepening)

In [ ]:
def alpha_beta(state: GameState,
                depth: int = 0,
                alpha: float = float('-inf'),
                beta: float = float('inf'),
                maximizing: bool = True,
                eval_fn: Optional[callable] = None,
                max_depth: float = float('inf')) -> Tuple[float, Optional[Any]]:
    """
    Alpha-Beta pruning implementation.
    
    Args:
        state: Current game state
        depth: Current depth in tree
        alpha: Best value MAX can guarantee (lower bound)
        beta: Best value MIN can guarantee (upper bound)
        maximizing: True if MAX's turn
        eval_fn: Evaluation function for depth limit
        max_depth: Maximum search depth
    
    Returns:
        (value, best_action) tuple
    """
    # Terminal or depth limit
    if state.is_terminal():
        player = 1 if maximizing else -1
        return state.utility(player), None
    
    if depth >= max_depth:
        if eval_fn:
            return eval_fn(state), None
        return 0, None
    
    player = state.current_player()
    legal_actions = state.get_legal_actions(player)
    
    if not legal_actions:
        return state.utility(player), None
    
    if maximizing:
        # MAX player
        value = float('-inf')
        best_action = None
        
        for action in legal_actions:
            successor = state.get_successor(action)
            child_value, _ = alpha_beta(successor, depth + 1, alpha, beta, 
                                        False, eval_fn, max_depth)
            
            if child_value > value:
                value = child_value
                best_action = action
            
            alpha = max(alpha, value)
            
            # Beta cutoff (prune)
            if beta <= alpha:
                break  # MIN won't choose this branch
        
        return value, best_action
    else:
        # MIN player
        value = float('inf')
        best_action = None
        
        for action in legal_actions:
            successor = state.get_successor(action)
            child_value, _ = alpha_beta(successor, depth + 1, alpha, beta,
                                        True, eval_fn, max_depth)
            
            if child_value < value:
                value = child_value
                best_action = action
            
            beta = min(beta, value)
            
            # Alpha cutoff (prune)
            if beta <= alpha:
                break  # MAX won't choose this branch
        
        return value, best_action


print('✓ Alpha-Beta pruning implemented')

## 3.5 Tic-Tac-Toe Implementation

Let's implement a complete game to test our algorithms.

**Tic-Tac-Toe Specifications:**
- 3x3 grid
- Players: X (1) and O (-1)
- Win: 3 in a row (horizontal, vertical, diagonal)
- Draw: Board full with no winner
- State space: 3^9 = 19,683 positions (including invalid)
- Game tree complexity: ~5,000 nodes

**Perfect Play Result**: Always draw with optimal play from both sides!

In [ ]:
class TicTacToe(GameState):
    """
    Tic-Tac-Toe game implementation.
    Player 1 (X) is MAX, Player -1 (O) is MIN.
    """
    
    def __init__(self, board=None, player=1):
        if board is None:
            self.board = np.zeros((3, 3), dtype=int)
        else:
            self.board = board.copy()
        self.player = player
    
    def get_legal_actions(self, player):
        """Return list of (row, col) tuples for empty cells."""
        return [(i, j) for i in range(3) for j in range(3) 
                if self.board[i, j] == 0]
    
    def get_successor(self, action):
        """Return new state after placing current player's mark."""
        new_state = TicTacToe(self.board, -self.player)
        new_state.board[action] = self.player
        return new_state
    
    def is_terminal(self) -> bool:
        """Check if game is over (win or draw)."""
        return self.get_winner() is not None or len(self.get_legal_actions(self.player)) == 0
    
    def get_winner(self):
        """Return winner (1, -1) or None if no winner yet."""
        # Check rows
        for i in range(3):
            if abs(self.board[i, :].sum()) == 3:
                return self.board[i, 0]
        
        # Check columns
        for j in range(3):
            if abs(self.board[:, j].sum()) == 3:
                return self.board[0, j]
        
        # Check diagonals
        if abs(np.trace(self.board)) == 3:
            return self.board[0, 0]
        if abs(np.trace(np.fliplr(self.board))) == 3:
            return self.board[0, 2]
        
        return None
    
    def utility(self, player) -> float:
        """Return utility from player's perspective."""
        winner = self.get_winner()
        if winner == player:
            return 1.0
        elif winner == -player:
            return -1.0
        return 0.0  # Draw
    
    def current_player(self) -> int:
        return self.player
    
    def display(self):
        """Pretty print the board."""
        symbols = {1: 'X', -1: 'O', 0: '.'}
        for i in range(3):
            print(' '.join(symbols[self.board[i, j]] for j in range(3)))
        print()


print('✓ Tic-Tac-Toe implemented')

In [ ]:
# Test Tic-Tac-Toe with Minimax and Alpha-Beta
print("=== Testing Tic-Tac-Toe ===")
print()

game = TicTacToe()
print("Initial board:")
game.display()

print("--- Minimax (complete search) ---")
start = time.time()
value, action = minimax(game, maximizing=True)
elapsed = time.time() - start
print(f"Best move: {action}")
print(f"Expected outcome: {value:.1f}")
print(f"Time: {elapsed:.4f}s")
print()

print("--- Alpha-Beta Pruning ---")
start = time.time()
value, action = alpha_beta(game, maximizing=True)
elapsed = time.time() - start
print(f"Best move: {action}")
print(f"Expected outcome: {value:.1f}")
print(f"Time: {elapsed:.4f}s")
print(f"Note: Alpha-Beta gives same result but faster!")
print()

# Show a few moves
game2 = game.get_successor(action)
print(f"After {action}:")
game2.display()

## 3.6 Monte Carlo Tree Search (MCTS)

MCTS is a **learning-based** approach that estimates game values through random simulations (rollouts).

### Why MCTS?

**Problems with Minimax/Alpha-Beta:**
1. Require accurate evaluation functions
2. Limited depth in large branching factor games
3. Assume we can evaluate non-terminal positions well

**MCTS Advantages:**
1. **No evaluation function needed** - learns from simulations
2. **Anytime algorithm** - improves with more time
3. **Handles large branching factors** - focuses on promising moves
4. **Asymmetric tree growth** - explores better moves more

### Four Phases

1. **Selection**: Navigate tree using UCT formula
2. **Expansion**: Add new node to tree
3. **Simulation**: Play random game to terminal state
4. **Backpropagation**: Update statistics in visited nodes

### UCT Formula

Upper Confidence Bound applied to Trees:

$$\text{UCT}(n) = \frac{w_n}{n_n} + c \sqrt{\frac{\ln N_n}{n_n}}$$

Where:
- $w_n$: wins from node n
- $n_n$: visits to node n
- $N_n$: visits to parent
- $c$: exploration constant (typically √2 ≈ 1.41)

**Exploitation** (first term): Prefer nodes with high win rate
**Exploration** (second term): Prefer less-visited nodes

### Why MCTS Works

1. **No domain knowledge needed** - learns through play
2. **Balances exploration/exploitation** via UCT
3. **Focuses computation** on relevant parts of tree
4. **Converges to minimax** with infinite simulations

### AlphaGo Connection

AlphaGo combined:
- MCTS for search
- Neural networks for evaluation and move selection
- This hybrid achieved superhuman performance in Go!

In [ ]:
class MCTSNode:
    """
    Node in Monte Carlo Tree Search.
    Stores state, statistics, parent, and children.
    """
    
    def __init__(self, state: GameState, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action  # Action that led to this node
        self.children = []
        
        # Statistics
        self.visits = 0
        self.wins = 0.0  # From perspective of parent's player
    
    def is_fully_expanded(self) -> bool:
        """Check if all legal actions have corresponding child nodes."""
        player = self.state.current_player()
        legal_actions = self.state.get_legal_actions(player)
        return len(self.children) == len(legal_actions)
    
    def best_child(self, c: float = 1.41):
        """
        Select best child using UCT formula.
        
        Args:
            c: Exploration constant (sqrt(2) by default)
        
        Returns:
            Child node with highest UCT value
        """
        def uct_value(node):
            if node.visits == 0:
                return float('inf')  # Prioritize unvisited
            
            exploitation = node.wins / node.visits
            exploration = c * math.sqrt(math.log(self.visits) / node.visits)
            return exploitation + exploration
        
        return max(self.children, key=uct_value)
    
    def expand(self):
        """
        Add a new child node for an untried action.
        
        Returns:
            Newly created child node
        """
        # Find untried actions
        tried_actions = {child.action for child in self.children}
        player = self.state.current_player()
        legal_actions = self.state.get_legal_actions(player)
        untried = [a for a in legal_actions if a not in tried_actions]
        
        # Pick random untried action
        action = random.choice(untried)
        next_state = self.state.get_successor(action)
        
        # Create and add child
        child = MCTSNode(next_state, parent=self, action=action)
        self.children.append(child)
        
        return child


def mcts(root_state: GameState, 
          num_simulations: int = 1000,
          c: float = 1.41,
          verbose: bool = True) -> Any:
    """
    Monte Carlo Tree Search implementation.
    
    Args:
        root_state: Initial game state
        num_simulations: Number of rollouts to perform
        c: UCT exploration constant
        verbose: Print progress
    
    Returns:
        Best action found
    """
    root = MCTSNode(root_state)
    root_player = root_state.current_player()
    
    start_time = time.time()
    
    for sim in range(num_simulations):
        node = root
        
        # 1. Selection: Navigate to leaf using UCT
        while node.is_fully_expanded() and not node.state.is_terminal():
            node = node.best_child(c)
        
        # 2. Expansion: Add new child if not terminal
        if not node.state.is_terminal() and not node.is_fully_expanded():
            node = node.expand()
        
        # 3. Simulation: Random playout from this node
        sim_state = TicTacToe(node.state.board, node.state.player)
        while not sim_state.is_terminal():
            player = sim_state.current_player()
            actions = sim_state.get_legal_actions(player)
            if not actions:
                break
            action = random.choice(actions)
            sim_state = sim_state.get_successor(action)
        
        # Get result from root player's perspective
        result = sim_state.utility(root_player)
        
        # 4. Backpropagation: Update statistics
        while node is not None:
            node.visits += 1
            # Convert result to [0,1] range (0=loss, 0.5=draw, 1=win)
            node.wins += (result + 1) / 2
            node = node.parent
            # Flip result for opponent
            result = -result
        
        if verbose and (sim + 1) % 250 == 0:
            print(f"Simulations: {sim + 1}/{num_simulations}")
    
    elapsed = time.time() - start_time
    
    if verbose:
        print(f"\nMCTS Complete:")
        print(f"  Simulations: {num_simulations}")
        print(f"  Time: {elapsed:.4f}s")
        print(f"  Sims/sec: {num_simulations/elapsed:.0f}")
    
    # Return most visited child (robust choice)
    best_child = max(root.children, key=lambda n: n.visits)
    
    if verbose:
        print(f"  Best action visits: {best_child.visits}")
        print(f"  Win rate: {best_child.wins/best_child.visits:.2%}")
    
    return best_child.action


print('✓ MCTS implemented')

In [ ]:
# Test MCTS
print("=== Testing MCTS on Tic-Tac-Toe ===")
print()

game_mcts = TicTacToe()
print("Initial board:")
game_mcts.display()

print("Running MCTS with 1000 simulations...")
best_action = mcts(game_mcts, num_simulations=1000)
print(f"\nMCTS best move: {best_action}")
print()

# Compare with Alpha-Beta
print("Comparing with Alpha-Beta:")
_, ab_action = alpha_beta(game_mcts, maximizing=True)
print(f"Alpha-Beta move: {ab_action}")
print()

if best_action == ab_action:
    print("✓ MCTS found the same move as Alpha-Beta!")
else:
    print("Different moves - both likely optimal for Tic-Tac-Toe")

## Programming Tasks

### Task 1: Connect Four (Medium)

Implement Connect Four game:
- 6x7 grid, pieces drop to lowest row
- Win: 4 in a row (horizontal/vertical/diagonal)
- Implement game state class
- Create evaluation function
- Test with Alpha-Beta at depth 6
- Compare Alpha-Beta vs MCTS

### Task 2: Minimax vs Alpha-Beta Analysis (Easy-Medium)

Compare algorithms empirically:
- Count nodes expanded in both algorithms
- Test on Tic-Tac-Toe at various positions
- Measure pruning effectiveness
- Visualize search tree sizes

### Task 3: Evaluation Function Design (Medium)

Design evaluation functions:
- For Connect Four
- Consider: center control, threats, potential wins
- Weight different features
- Test at different depths (4, 6, 8)
- Measure playing strength

### Task 4: MCTS Improvements (Hard)

Enhance MCTS:
- Implement progressive widening
- Add domain-specific rollout policy
- Use RAVE (Rapid Action Value Estimation)
- Compare performance vs vanilla MCTS

### Task 5: Game Playing Tournament (Hard)

Create agent tournament:
- Implement multiple agents (random, minimax, alpha-beta, MCTS)
- Run round-robin tournament
- Track wins/losses/draws
- Analyze which strategies work best
- Measure time per move

## Summary

This chapter explored adversarial search for two-player games:

### Algorithm Comparison

| Algorithm | Type | Time | Space | Optimality | Best For |
|-----------|------|------|-------|------------|----------|
| Minimax | Deductive | O(b^m) | O(bm) | Yes | Small games |
| Alpha-Beta | Deductive | O(b^(m/2)) | O(bm) | Yes | Chess-like games |
| MCTS | Inductive | Anytime | O(simulations) | Converges | Large branching, Go |

### Key Takeaways

1. **Minimax = Perfect Play**: Assumes both players play optimally
2. **Alpha-Beta = Smart Pruning**: Doubles effective search depth
3. **Move Ordering Matters**: Critical for Alpha-Beta efficiency
4. **MCTS = Learning**: No evaluation function needed
5. **Hybrid Approaches**: Modern AI combines search + learning (AlphaGo, AlphaZero)

### Practical Guidelines

**Use Minimax/Alpha-Beta When:**
- Can search to terminal states
- Have good evaluation function
- Moderate branching factor
- Need guaranteed optimal play

**Use MCTS When:**
- Large branching factor
- Hard to design evaluation function
- Can afford simulation time
- Asymmetric tree growth beneficial

**Modern Best Practice:**
- Combine MCTS with neural network evaluation (AlphaZero approach)
- Use MCTS for search, deep learning for position evaluation
- This gives best of both worlds!

### Historical Milestones

- **1997**: Deep Blue beats Kasparov (chess) - Alpha-Beta + evaluation
- **2016**: AlphaGo beats Lee Sedol (Go) - MCTS + deep learning
- **2017**: AlphaZero masters chess, shogi, Go - Self-play MCTS + RL

In the next chapter, we explore propositional logic and knowledge representation.

## Further Reading

### Textbooks
- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 3]
- Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). [Chapter 5]

### Seminal Papers
- Shannon, C. E. (1950). Programming a computer for playing chess. *Philosophical Magazine*.
- Knuth, D. E., & Moore, R. W. (1975). An analysis of alpha-beta pruning. *Artificial Intelligence*.
- Kocsis, L., & Szepesvári, C. (2006). Bandit based Monte-Carlo Planning. *ECML*.

### Modern Breakthroughs
- Silver, D., et al. (2016). Mastering the game of Go with deep neural networks. *Nature*.
- Silver, D., et al. (2017). Mastering Chess and Shogi by Self-Play. *arXiv*.

### Online Resources
- [Chess Programming Wiki](https://www.chessprogramming.org/)
- [AlphaGo Documentary](https://www.youtube.com/watch?v=WXuK6gekU1Y)
- [MCTS Survey](http://mcts.ai/)